# 🐍 Viper 2.0 (Exact 100M) — Tactical Terminal & DevOps Reflex Engine

**Viper 2.0** is an edge-native tactical reflex engine engineered from scratch with **exactly 100,000,000 locked parameters**.

### Core Specifications
- **Parameter Target**: Exactly `100,000,000` parameters (enforced by mathematical invariant)
- **Architecture**: 14 Transformer Layers ($d=800$, GQA 10/2, SwiGLU $d_{ff}=1920$, RoPE)
- **Vocabulary Size**: 17,451 ByteLevel BPE tokens
- **Resident RAM Budget**: < 105 MB peak resident RAM (`VmHWM` on CPU)
- **Quantized Size**: ~56.8 MB single self-contained Q4_0 GGUF binary
- **Latency**: 35ms – 50ms warm reflex on ordinary CPU
- **License**: Business Source License 1.1 (BSL 1.1) converting to Apache 2.0

## 1. Environment & GPU Verification
Run on Google Colab with a free **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

## 2. Clone Workspace & Install Dependencies

In [ ]:
!pip install -q tokenizers safetensors gguf
!git clone https://github.com/nio-labs/nio-xs-models.git
%cd nio-xs-models/models/viper-2/scripts
!ls -la

## 3. Verify Exact 100,000,000 Parameter Invariant

In [ ]:
from model import Viper2Config, Viper2ForCausalLM, count_parameters

config = Viper2Config()
model = Viper2ForCausalLM(config)
param_count = count_parameters(model)
print(f"Viper 2.0 Parameter Count: {param_count:,}")
assert param_count == 100_000_000, f"Expected 100,000,000, got {param_count}"
print("✅ Exact 100M Parameter Invariant Passed!")

## 4. Generate High-Entropy Training Curriculum (250,000 Samples)

In [ ]:
from dataset_generator import Viper2DatasetGenerator

gen = Viper2DatasetGenerator(seed=42)
gen.generate_curriculum(num_samples=250000, output_path="/content/viper2_curriculum_250k.jsonl")
!head -n 5 /content/viper2_curriculum_250k.jsonl

## 5. Train Dedicated 17,451 Vocabulary BPE Tokenizer

In [ ]:
from tokenizer_train import train_viper2_tokenizer

train_viper2_tokenizer(
    corpus_files=["/content/viper2_curriculum_250k.jsonl"],
    output_path="/content/viper-2-tokenizer.json",
    vocab_size=17451,
)

## 6. Train Viper 2.0 with Mixed Precision & Cosine LR Schedule

In [ ]:
!python train.py \
    --curriculum /content/viper2_curriculum_250k.jsonl \
    --tokenizer /content/viper-2-tokenizer.json \
    --out /content/checkpoints \
    --epochs 3 \
    --batch-size 32 \
    --lr 3e-4

## 7. Export to GGUF & Quantize to Q4_0 (~56.8 MB)

In [ ]:
!python export_gguf.py \
    --checkpoint /content/checkpoints/viper-2-final.pt \
    --tokenizer /content/viper-2-tokenizer.json \
    --out /content/viper-2.Q4_0.gguf

!ls -lh /content/viper-2.Q4_0.gguf

## 8. Download Trained Viper 2.0 Artifacts

In [ ]:
from google.colab import files
import os

for artifact in ["/content/viper-2.Q4_0.gguf", "/content/viper-2-tokenizer.json"]:
    if os.path.exists(artifact):
        files.download(artifact)
        print(f"Downloaded {artifact}")